In [2]:
import pandas as pd

input_path = "./phishing_legit_dataset_KD_10000.csv"
output_path = "./phishing_legit_dataset_KD_10000.jsonl"

df = pd.read_csv(input_path)
df.to_json(output_path, orient="records", lines=True, force_ascii=True)
print(f"Wrote {len(df)} records to {output_path}")

Wrote 10000 records to ./phishing_legit_dataset_KD_10000.jsonl


In [17]:
import pandas as pd

input_path = "./TREC_06.csv"
output_path = "./TREC_06.jsonl"

df = pd.read_csv(input_path)
df.to_json(output_path, orient="records", lines=True, force_ascii=True)
print(f"Wrote {len(df)} records to {output_path}")

ParserError: Error tokenizing data. C error: Buffer overflow caught - possible malformed input file.


In [14]:
import json
from pathlib import Path

input_path = Path("./PhishFuzzer_emails_entity_rephrased_v1.json")
spam_output_path = Path("./PhishFuzzer_emails_entity_rephrased_v1_spam.jsonl")
valid_output_path = Path("./PhishFuzzer_emails_entity_rephrased_v1_valid.jsonl")
phishing_output_path = Path("./PhishFuzzer_phishing.jsonl")

def load_records(path: Path):
    text = path.read_text(encoding="utf-8")
    try:
        data = json.loads(text)
        if isinstance(data, dict):
            for key in ("data", "records", "items"):
                if key in data and isinstance(data[key], list):
                    return data[key]
            raise ValueError("JSON object does not contain a list under common keys")
        if isinstance(data, list):
            return data
        raise ValueError("Unsupported JSON top-level type")
    except json.JSONDecodeError:
        # Fallback to JSONL
        return [json.loads(line) for line in text.splitlines() if line.strip()]

records = load_records(input_path)

def get_field(record, *keys):
    for key in keys:
        if key in record:
            return record[key]
    return None

def is_type(record, expected):
    value = str(get_field(record, "Type", "type", "label", "category")).strip().lower()
    return value == expected

spam_records = [r for r in records if is_type(r, "spam")]
valid_records = [r for r in records if is_type(r, "valid")]
phishing_records = [r for r in records if is_type(r, "phishing")]

def write_jsonl(path: Path, rows):
    with path.open("w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=True))
            f.write("\n")

write_jsonl(spam_output_path, spam_records)
write_jsonl(valid_output_path, valid_records)
write_jsonl(phishing_output_path, phishing_records)

print(f"Spam count: {len(spam_records)}")
print(f"Valid count: {len(valid_records)}")
print(f"Phishing count: {len(phishing_records)}")
print(f"Total records: {len(records)}")
print(f"Wrote spam records to {spam_output_path}")
print(f"Wrote valid records to {valid_output_path}")
print(f"Wrote phishing records to {phishing_output_path}")

Spam count: 6444
Valid count: 6600
Phishing count: 6756
Total records: 19800
Wrote spam records to PhishFuzzer_emails_entity_rephrased_v1_spam.jsonl
Wrote valid records to PhishFuzzer_emails_entity_rephrased_v1_valid.jsonl
Wrote phishing records to PhishFuzzer_phishing.jsonl


In [8]:
import json
from pathlib import Path

input_path = Path("./phishing_legit_dataset_KD_10000.jsonl")
output_path = Path("./phishing_legit_dataset_KD_10000_legitimate.jsonl")

records = [
    json.loads(line)
    for line in input_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
 ]

legitimate_records = [
    r
    for r in records
    if str(r.get("phishing_type", "")).strip().lower() == "legitimate"
 ]

with output_path.open("w", encoding="utf-8") as f:
    for r in legitimate_records:
        f.write(json.dumps(r, ensure_ascii=True))
        f.write("\n")

print(f"Legitimate count: {len(legitimate_records)}")
print(f"Total records: {len(records)}")
print(f"Wrote legitimate records to {output_path}")

Legitimate count: 4000
Total records: 10000
Wrote legitimate records to phishing_legit_dataset_KD_10000_legitimate.jsonl


In [22]:
import json
import re
from pathlib import Path

input_path = Path("./phishing_legitimate.jsonl")
output_path = Path("./phishing_legitimate1.jsonl")

subject_re = re.compile(r"^\s*Subject:\s*(.*?)\s*\n\s*\n", re.IGNORECASE | re.DOTALL)

with input_path.open("r", encoding="utf-8") as fin, output_path.open("w", encoding="utf-8") as fout:
    for line in fin:
        line = line.strip()
        if not line:
            continue
        obj = json.loads(line)

        text = obj.get("text", "")
        match = subject_re.match(text)
        if match:
            subject = match.group(1).strip()
            new_text = text[match.end():].lstrip("\n")
        else:
            subject = None
            new_text = text

        obj["subject"] = subject
        obj["text"] = new_text

        fout.write(json.dumps(obj, ensure_ascii=True))
        fout.write("\n")

print(f"Wrote: {output_path}")

Wrote: phishing_legitimate1.jsonl


In [10]:
import json
from pathlib import Path

input_path = Path("./CEAS_08.jsonl")
output_path = Path("./CEAS_08_sanitized.jsonl")

drop_fields = {"sender", "receiver", "date"}

with input_path.open("r", encoding="utf-8") as fin, output_path.open("w", encoding="utf-8") as fout:
    for line in fin:
        line = line.strip()
        if not line:
            continue
        obj = json.loads(line)
        for key in drop_fields:
            obj.pop(key, None)
        fout.write(json.dumps(obj, ensure_ascii=True))
        fout.write("\n")

print(f"Wrote: {output_path}")

Wrote: CEAS_08_sanitized.jsonl


In [13]:
import json
import re
from collections import Counter
from pathlib import Path

input_path = Path("./CEAS_08_sanitized.jsonl")
output_path = Path("./CEAS_08_sanitized_with_urls.jsonl")

url_re = re.compile(r"https?://[^\s\"'<>]+", re.IGNORECASE)
body_keys = ("body", "text", "content", "email", "message")

url_counts = Counter()

with input_path.open("r", encoding="utf-8") as fin, output_path.open("w", encoding="utf-8") as fout:
    for line in fin:
        line = line.strip()
        if not line:
            continue
        obj = json.loads(line)
        body = None
        for key in body_keys:
            if key in obj and isinstance(obj[key], str):
                body = obj[key]
                break
        urls = url_re.findall(body) if body else []
        if urls:
            url_counts.update(urls)
        obj["urls"] = urls
        obj["url_number"] = len(urls)
        fout.write(json.dumps(obj, ensure_ascii=True))
        fout.write("\n")

print(f"Unique URLs: {len(url_counts)}")
print(f"Wrote sanitized records with URLs to {output_path}")

Unique URLs: 27584
Wrote sanitized records with URLs to CEAS_08_sanitized_with_urls.jsonl


In [18]:
import pandas as pd

input_path = "./Nigerian_Fraud.csv"
output_path = "./Nigerian_Fraud.jsonl"

df = pd.read_csv(input_path)
df.to_json(output_path, orient="records", lines=True, force_ascii=True)
print(f"Wrote {len(df)} records to {output_path}")

Wrote 3332 records to ./Nigerian_Fraud.jsonl


In [19]:
import pandas as pd

input_path = "./Nazario.csv"
output_path = "./Nazario.jsonl"

df = pd.read_csv(input_path)
df.to_json(output_path, orient="records", lines=True, force_ascii=True)
print(f"Wrote {len(df)} records to {output_path}")

Wrote 1565 records to ./Nazario.jsonl


In [21]:
import json
from pathlib import Path

input_path = Path("./Nigerian_Fraud.jsonl")
output_path = Path("./Nigerian_Fraud_sanitized.jsonl")

drop_fields = {"sender", "receiver", "date"}

with input_path.open("r", encoding="utf-8") as fin, output_path.open("w", encoding="utf-8") as fout:
    for line in fin:
        line = line.strip()
        if not line:
            continue
        obj = json.loads(line)
        for key in drop_fields:
            obj.pop(key, None)
        fout.write(json.dumps(obj, ensure_ascii=True))
        fout.write("\n")

print(f"Wrote: {output_path}")

Wrote: Nigerian_Fraud_sanitized.jsonl


In [1]:
import json
import random
from pathlib import Path

input_path = Path("./phishing_legit_dataset_KD_10000.jsonl")
output_path = Path("./phishing.jsonl")
seed = 42
sample_size = 200

random.seed(seed)

def read_jsonl(path: Path):
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
    return rows

def is_label_one(value):
    if isinstance(value, str) and value.strip().isdigit():
        value = int(value.strip())
    return value == 1

rows = read_jsonl(input_path)
label_one_rows = [r for r in rows if is_label_one(r.get("label"))]

if len(label_one_rows) < sample_size:
    raise ValueError(
        f"Not enough label 1 rows: {len(label_one_rows)} available, {sample_size} requested"
    )

sampled = random.sample(label_one_rows, sample_size)

with output_path.open("w", encoding="utf-8") as f:
    for r in sampled:
        f.write(json.dumps(r, ensure_ascii=True))
        f.write("\n")

print(f"Wrote {len(sampled)} rows to {output_path}")

Wrote 200 rows to phishing.jsonl
